In [1]:
import htcondor
import os
import time
from utils.folder_setup import folder_setup

if not os.path.isdir("logs"):
    os.makedirs("logs/main")
    os.makedirs("logs/null")
    os.makedirs("logs/inference")

if not os.path.isdir("tmp"):
    os.mkdir("tmp")
    
if not os.path.isdir("Results"):
    os.mkdir("Results")

path = "/home/lfrahm/projects/pyALE"
if not os.path.isdir("Results/MainEffect/Full"):
    folder_setup(path, "MainEffect_Full")
    
path = "/home/lfrahm/projects/pyALE/"
file = "experiment_info.xlsx"
type_ = "M"
name = "Other"
tags1 = "+Other"
tags2 = "+I>C"


num_rep = 3000
clust_thresh = 0.001 

In [2]:
schedd = htcondor.Schedd()

main_wrap = htcondor.Submit({
    "executable": "$ENV(HOME)/projects/pyALE/me_wrap.sh",  # the program to run on the execute node
    "output": f"logs/main/{name}.out",       # anything the job prints to standard output will end up in this file
    "error": f"logs/main/{name}.err",        # anything the job prints to standard error will end up in this file
    "log": f"logs/main/{name}.log",          # this file will contain a record of what happened to the job
    "request_cpus": "1",            # how many CPU cores we want
    "request_memory": "2G",     # how much memory we want
    "arguments": f"{path} {file} {type_} {name} {tags1} {tags2}"
})
with schedd.transaction() as txn:
    main_id = main_wrap.queue(txn)

In [8]:
null_wrap = htcondor.Submit({
    "executable": "$ENV(HOME)/projects/pyALE/null_wrap.sh",  # the program to run on the execute node
    "output": f"logs/null/{name}_$(ProcId).out",       # anything the job prints to standard output will end up in this file
    "error": f"logs/null/{name}_$(ProcId).err",        # anything the job prints to standard error will end up in this file
    "log": f"logs/null/{name}_$(ProcId).log",          # this file will contain a record of what happened to the job
    "request_cpus": "1",            # how many CPU cores we want
    "request_memory": "2G",      # how much memory we want
    "arguments": f"{name} {clust_thresh} {e} {h}"
})


with schedd.transaction() as txn:
    null_id = null_wrap.queue(txn, count=num_rep//100)

In [9]:
inference_wrap = htcondor.Submit({
    "executable": "$ENV(HOME)/projects/pyALE/inference_wrap.sh",  # the program to run on the execute node
    "output": f"logs/inference/{name}.out",       # anything the job prints to standard output will end up in this file
    "error": f"logs/inference/{name}.err",        # anything the job prints to standard error will end up in this file
    "log": f"logs/inference/{name}.log",          # this file will contain a record of what happened to the job
    "request_cpus": "1",            # how many CPU cores we want
    "request_memory": "2G",      # how much memory we want
    "arguments": f"{name} {num_rep//100} {clust_thresh}"
})


with schedd.transaction() as txn:
    inference_id = inference_wrap.queue(txn)